In [1]:
# === FIX: Robust JSON parsing + Mistral retry + trusted citations ===
import json, re, os, requests, subprocess
from urllib.parse import urlparse

def extract_json_best_effort(s: str) -> dict:
    s = (s or "").strip()
    s = re.sub(r"^```(?:json)?\s*", "", s, flags=re.IGNORECASE)
    s = re.sub(r"\s*```$", "", s)
    if s.startswith("{") and s.endswith("}"):
        return json.loads(s)
    start = s.find("{")
    if start == -1:
        raise ValueError("No '{' found in model output.")
    depth = 0
    for i in range(start, len(s)):
        if s[i] == "{":
            depth += 1
        elif s[i] == "}":
            depth -= 1
            if depth == 0:
                return json.loads(s[start:i+1])
    raise ValueError("Could not parse JSON object.")

def mistral_chat_json(system_prompt: str, user_text: str, model: str, max_tokens: int = 800, retries: int = 1) -> dict:
    api_key = os.getenv("MISTRAL_API_KEY")
    if not api_key:
        raise ValueError("Missing MISTRAL_API_KEY.")
    url = "https://api.mistral.ai/v1/chat/completions"
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}
    strict_tail = "\n\nRETURN ONLY VALID JSON. No markdown. No commentary."
    last_text = None
    for attempt in range(retries + 1):
        payload = {
            "model": model,
            "temperature": 0.0,
            "max_tokens": max_tokens,
            "messages": [
                {"role": "system", "content": system_prompt + strict_tail},
                {"role": "user", "content": user_text},
            ],
        }
        r = requests.post(url, headers=headers, json=payload, timeout=90)
        if r.status_code != 200:
            raise RuntimeError(f"Mistral API error {r.status_code}: {r.text}")
        data = r.json()
        last_text = data["choices"][0]["message"]["content"]
        try:
            return extract_json_best_effort(last_text)
        except Exception:
            if attempt == retries:
                preview = (last_text or "")[:800]
                raise ValueError(f"Failed to parse JSON. Preview:\n{preview}")

TRUSTED_DOMAINS = {
    "who.int","nhs.uk","cdc.gov","nih.gov","ncbi.nlm.nih.gov",
    "acog.org","womenshealth.gov","mayoclinic.org",
    "clevelandclinic.org","healthdirect.gov.au"
}

def is_trusted(url: str) -> bool:
    if not url:
        return False
    host = (urlparse(url).netloc or "").lower().replace("www.","")
    return any(host == d or host.endswith("." + d) for d in TRUSTED_DOMAINS)


# WomenFalseInformation — Video Pipeline (Mistral-first, **requests-based**)

This notebook processes a **video** and produces:
- **Routing** (domain + risk + route) using **Mistral Large**
- **Claim extraction** using **Mistral Large**
- **Grounded evidence retrieval** using **Gemini + Google Search tool**
- **Final verdict JSON** using **Mistral Large**

✅ No `mistralai` SDK required (uses `requests.post`).

---

## Where to add the video path?
Go to **Cell V9** and set:

```python
VIDEO_PATH = "path/to/your_video.mp4"
MODE = "fallback"   # safest
```

Then run **Cell V10**.


## V1 — Install (optional) + Imports


In [3]:
# V1: Imports
# If you need installs, uncomment:
# !pip -q install -U google-genai opencv-python pillow requests

import os, json, re, hashlib
from pathlib import Path
import requests

from google import genai
from google.genai.types import GenerateContentConfig, Tool, GoogleSearch

print('✅ Imports loaded')


✅ Imports loaded


## V2 — Keys + Model names (NO hardcoding)


In [4]:
# V2: Keys + models
assert os.getenv('GEMINI_API_KEY'), '❌ Missing GEMINI_API_KEY'
assert os.getenv('MISTRAL_API_KEY'), '❌ Missing MISTRAL_API_KEY'

MISTRAL_MODEL = 'mistral-large-latest'
GEMINI_MODEL_TEXT   = 'gemini-2.5-pro'
GEMINI_MODEL_VISION = 'gemini-2.5-pro'

print('✅ Keys found')
print('🧠 Mistral model:', MISTRAL_MODEL)
print('🔎 Gemini model:', GEMINI_MODEL_TEXT)


✅ Keys found
🧠 Mistral model: mistral-large-latest
🔎 Gemini model: gemini-2.5-pro


## V3 — Create Gemini client


In [5]:
# V3: Gemini client
gclient = genai.Client(api_key=os.getenv('GEMINI_API_KEY'))
print('✅ Gemini client ready')


✅ Gemini client ready


## V4 — Helpers (hashing + JSON extraction)


In [6]:
# V4: Helpers

def sha256_bytes(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()

import json, re

import json, re

def extract_json_best_effort(s: str) -> dict:
    s = (s or "").strip()

    # Remove markdown fences if present
    s = re.sub(r"^```(?:json)?\s*", "", s, flags=re.IGNORECASE)
    s = re.sub(r"\s*```$", "", s)

    # Fast path
    if s.startswith("{") and s.endswith("}"):
        return json.loads(s)

    # Try to find the first JSON object by scanning braces
    start = s.find("{")
    if start == -1:
        raise ValueError("No '{' found in model output.")

    depth = 0
    for i in range(start, len(s)):
        if s[i] == "{":
            depth += 1
        elif s[i] == "}":
            depth -= 1
            if depth == 0:
                candidate = s[start:i+1]
                return json.loads(candidate)

    # If we got here, braces never balanced
    raise ValueError("Found '{' but could not parse a complete JSON object.")

## V5 — Mistral API caller (requests-based)


In [7]:
import os, requests, json

def mistral_chat_json(system_prompt: str, user_text: str, model: str, max_tokens: int = 800, retries: int = 1) -> dict:
    api_key = os.getenv("MISTRAL_API_KEY")
    if not api_key:
        raise ValueError("Missing MISTRAL_API_KEY.")

    url = "https://api.mistral.ai/v1/chat/completions"
    headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}

    base_system = system_prompt.strip()
    strict_tail = "\n\nRETURN ONLY VALID JSON. No markdown. No commentary. No extra keys."

    last_err = None
    last_text = None

    for attempt in range(retries + 1):
        sp = base_system + strict_tail
        if attempt > 0:
            sp += "\nIf you previously output non-JSON, FIX IT and output ONLY JSON now."

        payload = {
            "model": model,
            "temperature": 0.0,
            "max_tokens": max_tokens,
            "messages": [
                {"role": "system", "content": sp},
                {"role": "user", "content": user_text},
            ],
        }

        r = requests.post(url, headers=headers, json=payload, timeout=90)
        if r.status_code != 200:
            raise RuntimeError(f"Mistral API error {r.status_code}: {r.text}")

        data = r.json()
        last_text = data["choices"][0]["message"]["content"]

        try:
            return extract_json_best_effort(last_text)
        except Exception as e:
            last_err = e

    # If all retries fail, show a snippet for debugging
    preview = (last_text or "")[:800]
    raise ValueError(f"Failed to parse JSON after retries. Last error: {last_err}\nModel output preview:\n{preview}")

## V6 — Schemas


In [8]:
# V6: Schemas
ROUTER_SCHEMA = {
  'type': 'object',
  'properties': {
    'domain': {'type':'string'},
    'risk_level': {'type':'string', 'enum':['low','medium','high']},
    'route': {'type':'string', 'enum':['fast_check','deep_verify']},
    'reason': {'type':'string'}
  },
  'required': ['domain','risk_level','route','reason']
}

CLAIMS_SCHEMA = {
  'type':'object',
  'properties':{
    'claims':{
      'type':'array',
      'items':{
        'type':'object',
        'properties':{
          'text':{'type':'string'},
          'type':{'type':'string'},
          'topic':{'type':'string'}
        },
        'required':['text']
      }
    }
  },
  'required':['claims']
}

VERIFY_SCHEMA = {
  'type':'object',
  'properties':{
    'verdict':{'type':'string','enum':['true','false','misleading','uncertain']},
    'confidence':{'type':'number'},
    'rationale':{'type':'string'},
    'citations':{'type':'array','items':{'type':'string'}}
  },
  'required':['verdict','confidence','rationale','citations']
}

print('✅ Schemas loaded')


✅ Schemas loaded


## V7 — Video → text adapter (preferred + fallback)


**Preferred:** Gemini file upload → transcript/narration.  
**Fallback (safest):** sample frames → Gemini Vision → pseudo-transcript.


In [9]:
# V7A: Preferred (SDK-dependent)

def analyze_video_with_gemini_file(video_path: str) -> dict:
    print('🎞️ Video input:', video_path)
    video_bytes = Path(video_path).read_bytes()
    print('size (MB):', round(len(video_bytes)/1e6, 2))

    up = gclient.files.upload(path=video_path)

    prompt = (
        'Analyze this social media video for misinformation.\n\n'
        'Return STRICT JSON with:\n'
        '- transcript: best-effort transcript (or detailed narration if transcript unavailable)\n'
        '- key_points: 5-12 bullets of what is claimed/shown\n'
        '- language: primary language\n'
        '- notes: uncertainties\n\n'
        'Output ONLY valid JSON.'
    )

    resp = gclient.models.generate_content(
        model=GEMINI_MODEL_TEXT,
        contents=[prompt, up],
        config=GenerateContentConfig(temperature=0.0, response_mime_type='application/json'),
    )

    out = extract_json_best_effort(resp.text)
    t = out.get('transcript','')
    print('✅ transcript/narration (first 350 chars):')
    print(t[:350] + ('...' if len(t) > 350 else ''))
    return out


In [10]:
# V7B: Fallback

import cv2
from PIL import Image

def sample_video_frames(video_path: str, every_seconds: float = 2.0, max_frames: int = 16):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    step = max(1, int(fps * every_seconds))

    frames = []
    idx = 0
    grabbed = 0

    while cap.isOpened() and grabbed < max_frames:
        ret, frame = cap.read()
        if not ret:
            break
        if idx % step == 0:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(Image.fromarray(rgb))
            grabbed += 1
        idx += 1

    cap.release()
    return frames

def describe_frames_with_gemini(frames):
    descriptions = []
    for i, img in enumerate(frames):
        prompt = (
            f'Describe frame #{i+1} of a social media video.\n'
            'Include any visible text (verbatim if possible) and what is happening.\n'
            'Keep it concise.'
        )
        resp = gclient.models.generate_content(
            model=GEMINI_MODEL_VISION,
            contents=[prompt, img],
            config=GenerateContentConfig(temperature=0.0),
        )
        desc = (resp.text or '').strip()
        print(f'✅ Frame {i+1} desc (first 160):', desc[:160] + ('...' if len(desc) > 160 else ''))
        descriptions.append(desc)

    return '\n\n'.join([f'FRAME {i+1}: {d}' for i, d in enumerate(descriptions)])

def analyze_video_fallback(video_path: str) -> dict:
    print('🎞️ Video input:', video_path)
    frames = sample_video_frames(video_path)
    print('✅ Sampled frames:', len(frames))

    pseudo = describe_frames_with_gemini(frames)
    return {
        'transcript': pseudo,
        'key_points': [],
        'language': 'unknown',
        'notes': 'Fallback mode: pseudo-transcript built from sampled frames.'
    }


## V8 — Mistral-first reasoning: routing + claims + verdict


In [11]:
# V8: Prompts + functions

ROUTER_SYSTEM = (
    'You are a routing classifier for a misinformation detection pipeline.\n'
    'Given content, decide domain + risk and whether we should do deep verification.\n\n'
    'Rules:\n'
    '- High risk: health, safety, legal, financial fraud, incitement, elections\n'
    '- Medium: strong factual claims likely to mislead people\n'
    '- Low: opinions, jokes, personal stories without factual assertions\n\n'
    'Return STRICT JSON matching the provided schema.'
)

CLAIM_SYSTEM = (
    'You extract CHECKABLE factual claims from transcripts / scene descriptions.\n'
    'Claims should be verifiable statements, not opinions.\n'
    'Return STRICT JSON matching the provided schema.'
)

VERDICT_SYSTEM = (
    'You are a fact-checking assistant.\n'
    'Given a claim and grounded evidence, output a strict JSON verdict.\n'
    'Use only the evidence provided. If evidence is insufficient, set verdict=uncertain with low confidence.'
)

def route_with_mistral(transcript_text: str) -> dict:
    print('\n🚦 Router input (first 300 chars):')
    print(transcript_text[:300] + ('...' if len(transcript_text) > 300 else ''))

    return mistral_chat_json(
        system_prompt=ROUTER_SYSTEM + '\n\nSchema:\n' + json.dumps(ROUTER_SCHEMA),
        user_text=transcript_text,
        model=MISTRAL_MODEL,
        max_tokens=400
    )

def extract_claims_with_mistral(transcript_text: str) -> dict:
    print('\n🧾 Claim extraction input (first 300 chars):')
    print(transcript_text[:300] + ('...' if len(transcript_text) > 300 else ''))

    return mistral_chat_json(
        system_prompt=CLAIM_SYSTEM + '\n\nSchema:\n' + json.dumps(CLAIMS_SCHEMA),
        user_text=transcript_text,
        model=MISTRAL_MODEL,
        max_tokens=700
    )

def get_grounded_evidence_with_gemini(claim_text: str) -> str:
    prompt = (
        'Use Google Search grounding to find evidence about this claim.\n\n'
        f'Claim:\n{claim_text}\n\n'
        'Return: bullet points summarizing evidence and include source titles + URLs in plain text.'
    )

    print('\n🔎 Grounding input (Gemini tools):', claim_text)

    resp = gclient.models.generate_content(
        model=GEMINI_MODEL_TEXT,
        contents=prompt,
        config=GenerateContentConfig(
            temperature=0.0,
            tools=[Tool(google_search=GoogleSearch())],
        ),
    )

    evidence = (resp.text or '').strip()
    print('✅ Evidence (first 350 chars):')
    print(evidence[:350] + ('...' if len(evidence) > 350 else ''))
    return evidence

def synthesize_verdict_with_mistral(claim_text: str, grounded_evidence: str) -> dict:
    user = (
        f'Claim:\n{claim_text}\n\n'
        'Grounded evidence (with URLs):\n'
        f'{grounded_evidence}\n\n'
        'Return STRICT JSON matching the schema.\n'
        'citations must be a list of URLs found in the evidence.'
    )

    out = mistral_chat_json(
        system_prompt=VERDICT_SYSTEM + '\n\nSchema:\n' + json.dumps(VERIFY_SCHEMA),
        user_text=user,
        model=MISTRAL_MODEL,
        max_tokens=700
    )

    print('✅ Verdict JSON (Mistral):', out)
    return out

def verify_claim_mistral_first(claim_text: str) -> dict:
    evidence = get_grounded_evidence_with_gemini(claim_text)
    return synthesize_verdict_with_mistral(claim_text, evidence)


## V9 — Configure your video path (EDIT THIS CELL)


In [12]:
# V9: 🔥 EDIT THIS CELL
VIDEO_PATH = r'C:\Users\absax\OneDrive\Documents\Personal\Project\Women False Information\MisinformationVideo.mp4'
MODE = 'fallback'  # safest

print('VIDEO_PATH:', VIDEO_PATH)
print('MODE:', MODE)


VIDEO_PATH: C:\Users\absax\OneDrive\Documents\Personal\Project\Women False Information\MisinformationVideo.mp4
MODE: fallback


## V10 — Run end-to-end pipeline


In [13]:
# V10: Run pipeline (FIXED)

from pathlib import Path

def process_video(video_path: str, mode: str = "fallback") -> dict:
    # (optional) hash for caching / logging
    b = Path(video_path).read_bytes()
    file_hash = sha256_bytes(b)

    # 1) Video -> transcript/text
    vinfo = analyze_video_with_gemini_file(video_path) if mode == "file" else analyze_video_fallback(video_path)
    transcript = (vinfo.get("transcript") or "").strip()

    print("\n✅ Transcript preview (first 400 chars):")
    print(transcript[:400] + ("..." if len(transcript) > 400 else ""))

    # 2) Mistral routing + claim extraction
    routing = route_with_mistral(transcript)
    print("\n✅ Routing output:")
    print(json.dumps(routing, indent=2))

    claims_out = extract_claims_with_mistral(transcript)
    claims = claims_out.get("claims", [])  # ✅ FIX: define claims from claims_out
    print("\n✅ Claims found:", len(claims))
    for i, c in enumerate(claims, 1):
        print(f"{i}. {c.get('text','')}")

    # 3) Verify per claim
    do_deep = (routing.get("route") == "deep_verify")
    per_claim_results = []

    for c in claims:
        claim_text = (c.get("text") or "").strip()
        if not claim_text:
            continue

        print("\n" + "=" * 60)
        print("VERIFYING:", claim_text)

        if do_deep:
            vres = verify_claim_mistral_first(claim_text)
        else:
            vres = {
                "verdict": "uncertain",
                "confidence": 0.3,
                "rationale": "Fast route: grounding skipped.",
                "citations": []
            }

        # ✅ Add trusted citations (safe even if citations missing)
        vres["trusted_citations"] = [u for u in vres.get("citations", []) if is_trusted(u)]

        per_claim_results.append({
            "claim": c,
            "verification": vres
        })

    # 4) Return final object
    return {
        "video_path": video_path,
        "file_hash": file_hash,
        "mode": mode,
        "video_info": vinfo,
        "routing": routing,
        "claims": claims_out,
        "results": per_claim_results
    }

output = process_video(VIDEO_PATH, mode=MODE)
print("\n✅ FINAL OUTPUT keys:", list(output.keys()))
print("✅ Number of claims:", len(output.get("claims", {}).get("claims", [])))
print("✅ Verified claims:", len(output.get("results", [])))

🎞️ Video input: C:\Users\absax\OneDrive\Documents\Personal\Project\Women False Information\MisinformationVideo.mp4
✅ Sampled frames: 5
✅ Frame 1 desc (first 160): The first frame is a solid black screen with no text or visible action.
✅ Frame 2 desc (first 160): In a close-up shot, a woman with dark hair and a slight, friendly smile looks directly at the camera. She is wearing a brown collared shirt and stands against a...
✅ Frame 3 desc (first 160): In this close-up shot, a woman with dark hair and a brown collared shirt smiles warmly at the camera. She is positioned against a solid, dark teal background. T...
✅ Frame 4 desc (first 160): In this close-up shot, a woman with dark hair and a brownish-grey collared shirt is smiling gently at the camera. She is in front of a solid dark teal backgroun...
✅ Frame 5 desc (first 160): In this close-up shot, a woman with dark hair and brown eyes looks directly at the camera with a slight, pleasant smile. She is wearing a brownish-gray collared.

## V11 — Pretty print final output (optional)


In [14]:
print(json.dumps(output, indent=2)[:4000] + ('\n... (truncated)' if len(json.dumps(output)) > 4000 else ''))


{
  "video_path": "C:\\Users\\absax\\OneDrive\\Documents\\Personal\\Project\\Women False Information\\MisinformationVideo.mp4",
  "file_hash": "664fd4b5680283c76896b605c9d521932a050307c47287d65c5dea020e358d6a",
  "mode": "fallback",
  "video_info": {
    "transcript": "FRAME 1: The first frame is a solid black screen with no text or visible action.\n\nFRAME 2: In a close-up shot, a woman with dark hair and a slight, friendly smile looks directly at the camera. She is wearing a brown collared shirt and stands against a solid dark teal background. There is no visible text.\n\nFRAME 3: In this close-up shot, a woman with dark hair and a brown collared shirt smiles warmly at the camera. She is positioned against a solid, dark teal background. There is no visible text.\n\nFRAME 4: In this close-up shot, a woman with dark hair and a brownish-grey collared shirt is smiling gently at the camera. She is in front of a solid dark teal background. There is no visible text.\n\nFRAME 5: In this clos

In [15]:
# Print ONLY models that support generateContent
ok = []
for m in gclient.models.list():
    name = getattr(m, "name", None)
    methods = getattr(m, "supported_generation_methods", None) or []
    if name and ("generateContent" in methods):
        ok.append(name)

print("generateContent models:", len(ok))
for n in ok:
    print("✅", n)

generateContent models: 0


In [16]:
# !pip -q install -U requests

import os, json, time, subprocess
from pathlib import Path
import requests

FFMPEG_EXE = r"C:\Users\absax\AppData\Local\Microsoft\WinGet\Packages\Gyan.FFmpeg_Microsoft.Winget.Source_8wekyb3d8bbwe\ffmpeg-8.0.1-full_build\bin\ffmpeg.exe"

In [17]:
ELEVEN_API_KEY = os.getenv("ELEVENLABS_API_KEY") or "sk_4435949177b39eb24d7bb20a620ebbbe356cc37a9208df02"
assert ELEVEN_API_KEY, "Missing ELEVENLABS_API_KEY (or XI_API_KEY)"

EL_HEADERS = {"xi-api-key": ELEVEN_API_KEY}

def run(cmd):
    print("▶", " ".join(cmd))
    subprocess.check_call(cmd)

In [18]:
import subprocess
from pathlib import Path

def extract_audio(video_path: str, out_wav: str = "tmp_audio.wav") -> str:
    """
    Extract audio from video using FFmpeg
    Output → mono 16k WAV (best for speech-to-text)
    """

    if not Path(FFMPEG_EXE).exists():
        raise FileNotFoundError(f"FFmpeg not found:\n{FFMPEG_EXE}")

    if not Path(video_path).exists():
        raise FileNotFoundError(f"Video not found:\n{video_path}")

    cmd = [
        FFMPEG_EXE,
        "-y",
        "-i", video_path,
        "-vn",
        "-ac", "1",
        "-ar", "16000",
        "-c:a", "pcm_s16le",
        out_wav,
    ]

    print("▶ Running FFmpeg")
    subprocess.check_call(cmd)

    print("✅ Audio extracted:", out_wav)
    return out_wav

In [19]:
from pathlib import Path
print(Path(FFMPEG_EXE).exists())

True


In [20]:
from pathlib import Path
import requests

# Try v2 first; if your account only supports v1, it will fall back automatically.
ELEVEN_STT_MODEL_PRIMARY = "scribe_v2"
ELEVEN_STT_MODEL_FALLBACK = "scribe_v1"

def eleven_transcribe(file_path: str) -> dict:
    """
    Input: audio/video file
    Output: dict containing text + language_code (and more)
    """
    url = "https://api.elevenlabs.io/v1/speech-to-text"

    def _call(model_id: str):
        with open(file_path, "rb") as f:
            files = {"file": (Path(file_path).name, f)}
            data = {"model_id": model_id}  # ✅ REQUIRED (your 422 proves it)
            return requests.post(url, headers=EL_HEADERS, files=files, data=data, timeout=300)

    r = _call(ELEVEN_STT_MODEL_PRIMARY)
    if r.status_code == 422 and "model_id" in r.text:
        # shouldn't happen now, but keep guard
        pass

    # If the model name is not accepted, fallback
    if r.status_code != 200:
        r2 = _call(ELEVEN_STT_MODEL_FALLBACK)
        if r2.status_code != 200:
            raise RuntimeError(
                f"Eleven STT error primary={r.status_code}: {r.text}\n"
                f"Eleven STT error fallback={r2.status_code}: {r2.text}"
            )
        r = r2

    out = r.json()
    print("✅ STT model:", out.get("model_id", "unknown"))
    print("✅ language_code:", out.get("language_code"), "| prob:", out.get("language_probability"))
    print("✅ text (first 250):", (out.get("text","")[:250] + ("..." if len(out.get("text","")) > 250 else "")))
    return out

In [21]:
def eleven_create_dub_to_english(source_file_path: str, source_lang: str = "auto") -> str:
    url = "https://api.elevenlabs.io/v1/dubbing"
    with open(source_file_path, "rb") as f:
        files = {"file": (Path(source_file_path).name, f)}
        data = {
            "name": f"dub_to_en_{int(time.time())}",
            "source_lang": source_lang,   # "auto" allowed :contentReference[oaicite:9]{index=9}
            "target_lang": "en",          # required :contentReference[oaicite:10]{index=10}
            "mode": "automatic"
        }
        r = requests.post(url, headers=EL_HEADERS, files=files, data=data, timeout=300)

    if r.status_code != 200:
        raise RuntimeError(f"Eleven dubbing create error {r.status_code}: {r.text}")

    dubbing_id = r.json()["dubbing_id"]
    print("✅ dubbing_id:", dubbing_id)
    return dubbing_id

def eleven_wait_for_dub(dubbing_id: str, poll_sec: int = 5, timeout_sec: int = 600) -> dict:
    url = f"https://api.elevenlabs.io/v1/dubbing/{dubbing_id}"
    t0 = time.time()
    while True:
        r = requests.get(url, headers=EL_HEADERS, timeout=30)
        if r.status_code != 200:
            raise RuntimeError(f"Eleven dubbing status error {r.status_code}: {r.text}")
        meta = r.json()
        status = meta.get("status")
        print("… status:", status)
        if status in ("dubbed", "failed"):
            return meta
        if time.time() - t0 > timeout_sec:
            raise TimeoutError("Dubbing timed out")
        time.sleep(poll_sec)

def eleven_download_dubbed_audio(dubbing_id: str, language_code: str = "en", out_path: str = "dubbed_en.mp3") -> str:
    # Endpoint per docs search result: /v1/dubbing/:dubbing_id/audio/:language_code :contentReference[oaicite:11]{index=11}
    url = f"https://api.elevenlabs.io/v1/dubbing/{dubbing_id}/audio/{language_code}"
    r = requests.get(url, headers=EL_HEADERS, timeout=300)
    if r.status_code != 200:
        raise RuntimeError(f"Eleven get dubbed audio error {r.status_code}: {r.text}")
    Path(out_path).write_bytes(r.content)
    print("✅ Saved dubbed audio:", out_path, "| bytes:", len(r.content))
    return out_path

In [22]:
def normalize_lang(code: str) -> str:
    """
    Normalize language codes to ISO-639-1-ish where possible.
    Examples:
      eng -> en
      en-US -> en
      hin -> hi
    """
    if not code:
        return ""
    c = code.strip().lower()
    # common 3-letter to 2-letter mappings we care about
    mapping = {
        "eng": "en",
        "hin": "hi",
        "spa": "es",
        "fra": "fr",
        "deu": "de",
        "ger": "de",
        "ita": "it",
        "por": "pt",
        "ara": "ar",
        "rus": "ru",
        "jpn": "ja",
        "kor": "ko",
        "zho": "zh",
    }
    # collapse regional variants
    c = c.split("-")[0].split("_")[0]
    return mapping.get(c, c)

In [23]:
def video_to_english_transcript(video_path: str) -> dict:
    wav_path = extract_audio(video_path, out_wav="tmp_audio.wav")

    stt0 = eleven_transcribe(wav_path)
    lang_raw = stt0.get("language_code") or ""
    lang = normalize_lang(lang_raw)
    text0 = stt0.get("text", "")

    # ✅ If already English, do NOT dub
    if lang == "en":
        return {"original_stt": stt0, "english_text": text0, "used_dubbing": False}

    # If not English, THEN dub + re-transcribe (optional)
    dubbing_id = eleven_create_dub_to_english(wav_path, source_lang="auto")  # note: use audio, not mp4
    meta = eleven_wait_for_dub(dubbing_id)
    if meta.get("status") != "dubbed":
        raise RuntimeError(f"Dubbing failed: {meta}")

    dubbed_audio = eleven_download_dubbed_audio(dubbing_id, "en", out_path="dubbed_en.mp3")
    stt_en = eleven_transcribe(dubbed_audio)

    return {"original_stt": stt0, "english_text": stt_en.get("text",""), "used_dubbing": True}

In [24]:
v = video_to_english_transcript(VIDEO_PATH)
transcript = v["english_text"]

print("\n✅ USING AUDIO TRANSCRIPT (first 600 chars):")
print(transcript[:600] + ("..." if len(transcript) > 600 else ""))

▶ Running FFmpeg
✅ Audio extracted: tmp_audio.wav
✅ STT model: unknown
✅ language_code: eng | prob: 0.9851394891738892
✅ text (first 250): You can't get pregnant if you have PCOS. PCOS means you'll never lose weight. If you're not overweight, you don't have PCOS. Birth control cures PCOS

✅ USING AUDIO TRANSCRIPT (first 600 chars):
You can't get pregnant if you have PCOS. PCOS means you'll never lose weight. If you're not overweight, you don't have PCOS. Birth control cures PCOS


In [25]:
TRUSTED_DOMAINS = ["who.int", "nhs.uk", "cdc.gov", "nih.gov", "ncbi.nlm.nih.gov", "acog.org", "womenshealth.gov", "clevelandclinic.org", "mayoclinic.org"]

def is_trusted(url: str) -> bool:
    u = (url or "").lower()
    return any(d in u for d in TRUSTED_DOMAINS)

In [26]:
vres["trusted_citations"] = [u for u in vres.get("citations", []) if is_trusted(u)]

NameError: name 'vres' is not defined

In [ ]:
# --- Misinformation pipeline on AUDIO transcript ---

# A) Routing (Mistral)
routing = route_with_mistral(transcript)
print("\n🚦 ROUTING:")
print(json.dumps(routing, indent=2))

# B) Claim extraction (Mistral)
claims_out = extract_claims_with_mistral(transcript)
claims = claims_out.get("claims", [])
print("\n🧾 CLAIMS FOUND:", len(claims))
for i, c in enumerate(claims, 1):
    print(f"{i}. {c.get('text','')}")

# C) Verify each claim (Gemini grounding + Mistral verdict)
do_deep = (routing.get("route") == "deep_verify")

per_claim_results = []
for i, c in enumerate(claims, 1):
    claim_text = (c.get("text") or "").strip()
    if not claim_text:
        continue

    print("\n" + "="*60)
    print(f"VERIFYING CLAIM {i}: {claim_text}")

    if do_deep:
        vres = verify_claim_mistral_first(claim_text)
    else:
        vres = {
            "verdict": "uncertain",
            "confidence": 0.3,
            "rationale": "Fast route: grounding skipped.",
            "citations": []
        }

    print("✅ VERDICT:", vres.get("verdict"), "| conf:", vres.get("confidence"))
    per_claim_results.append({"claim": c, "verification": vres})

# D) Final combined object
final_output = {
    "video_path": VIDEO_PATH,
    "stt_language_code": v.get("original_stt", {}).get("language_code"),
    "transcript": transcript,
    "routing": routing,
    "claims": claims_out,
    "results": per_claim_results
}

print("\n✅ DONE. Results:", len(per_claim_results))



🚦 Router input (first 300 chars):
You can't get pregnant if you have PCOS. PCOS means you'll never lose weight. If you're not overweight, you don't have PCOS. Birth control cures PCOS

🚦 ROUTING:
{
  "domain": "health",
  "risk_level": "high",
  "route": "deep_verify",
  "reason": "The content contains multiple false and harmful medical claims about PCOS (Polycystic Ovary Syndrome) that could mislead individuals about their health, fertility, and treatment options. These claims are high-risk as they may influence medical decisions and well-being."
}

🧾 Claim extraction input (first 300 chars):
You can't get pregnant if you have PCOS. PCOS means you'll never lose weight. If you're not overweight, you don't have PCOS. Birth control cures PCOS

🧾 CLAIMS FOUND: 4
1. You can't get pregnant if you have PCOS.
2. PCOS means you'll never lose weight.
3. If you're not overweight, you don't have PCOS.
4. Birth control cures PCOS.

VERIFYING CLAIM 1: You can't get pregnant if you have PCOS.

🔎 Gr

ValueError: No JSON found in output.

## ⭐ Timestamped misinformation (segment-level)

In [ ]:
import math
from pathlib import Path

def get_audio_duration_sec(audio_path: str) -> float:
    ffprobe_exe = str(Path(FFMPEG_EXE).with_name("ffprobe.exe"))
    cmd = [ffprobe_exe, "-v", "error", "-show_entries", "format=duration",
           "-of", "default=noprint_wrappers=1:nokey=1", audio_path]
    out = subprocess.check_output(cmd, text=True).strip()
    return float(out)

def split_audio_into_segments(audio_path: str, segment_sec: int = 8, out_dir: str = "audio_segments"):
    Path(out_dir).mkdir(parents=True, exist_ok=True)
    dur = get_audio_duration_sec(audio_path)
    n = math.ceil(dur / segment_sec)
    segs = []
    for i in range(n):
        start = i * segment_sec
        end = min((i+1)*segment_sec, dur)
        out_path = str(Path(out_dir)/f"seg_{i:03d}.wav")
        subprocess.check_call([
            FFMPEG_EXE,"-y","-ss",str(start),"-t",str(end-start),
            "-i",audio_path,"-ac","1","-ar","16000","-c:a","pcm_s16le",out_path
        ])
        segs.append({"path":out_path,"start":start,"end":end})
    return segs

def fmt_ts(sec):
    m=int(sec//60); s=int(sec%60)
    return f"{m:02d}:{s:02d}"


In [ ]:
def process_video_with_timestamps(video_path, segment_sec=8):
    wav_path = extract_audio(video_path, out_wav="tmp_audio.wav")
    segs = split_audio_into_segments(wav_path, segment_sec)

    timeline=[]
    for seg in segs:
        stt = eleven_transcribe(seg["path"])
        text = (stt.get("text") or "").strip()
        if not text:
            continue

        routing = route_with_mistral(text)
        claims_out = extract_claims_with_mistral(text)
        claims = claims_out.get("claims",[])
        do_deep = routing.get("route")=="deep_verify"

        for c in claims:
            claim=(c.get("text") or "").strip()
            if not claim:
                continue
            vres = verify_claim_mistral_first(claim) if do_deep else {"verdict":"uncertain","confidence":0.3,"citations":[]}
            vres["trusted_citations"]=[u for u in vres.get("citations",[]) if is_trusted(u)]
            timeline.append({
                "time":f"{fmt_ts(seg['start'])}–{fmt_ts(seg['end'])}",
                "claim":claim,
                "verdict":vres.get("verdict"),
                "confidence":vres.get("confidence")
            })

    timeline.sort(key=lambda r:r["time"])
    for r in timeline:
        print(r["time"],"|",r["verdict"],"|",r["claim"])
    return timeline

TS_OUTPUT = process_video_with_timestamps(VIDEO_PATH, segment_sec=8)
